In [9]:
!pip install -q anomalib lightning openvino opencv-python-headless pillow

In [10]:
from pathlib import Path

import torch
from anomalib.data import Folder
from anomalib.data.utils.split import TestSplitMode, ValSplitMode
from anomalib.engine import Engine
from anomalib.models import Patchcore
from anomalib.engine import Engine
import warnings
warnings.filterwarnings("ignore")


In [11]:
# Change this to the real path you found with !find
normal_dir = Path("G01_aligned_last500/train/normal")

angle = "G01"
model_set = "RembgAlignedPatchcore"

image_size = 256
batch_size = 32
num_workers = 2

backbone = "wide_resnet50_2"
layers = ("layer2", "layer3")
coreset_sampling_ratio = 0.1
num_neighbors = 9
normal_split_ratio = 0.2
seed = 42

results_dir = Path("working/results")
models_dir = Path("working/models")

image_count = sum(
    1 for path in normal_dir.rglob("*")
    if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
)

print("Normal dir:", normal_dir)
print("Exists:", normal_dir.exists())
print("Images:", image_count)

if not normal_dir.exists():
    raise FileNotFoundError(normal_dir)

if image_count == 0:
    raise RuntimeError("No images found.")


Normal dir: G01_aligned_last500/train/normal
Exists: True
Images: 3491


In [12]:
accelerator = "gpu" if torch.cuda.is_available() else "cpu"

print("Accelerator:", accelerator)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pre_processor = Patchcore.configure_pre_processor(
    image_size=(image_size, image_size)
)

model = Patchcore(
    backbone=backbone,
    layers=layers,
    pre_trained=True,
    coreset_sampling_ratio=coreset_sampling_ratio,
    num_neighbors=num_neighbors,
    pre_processor=pre_processor,
)

datamodule = Folder(
    name=angle,
    normal_dir=normal_dir,
    train_batch_size=batch_size,
    eval_batch_size=batch_size,
    num_workers=num_workers,
    normal_split_ratio=normal_split_ratio,
    test_split_mode=TestSplitMode.NONE,
    val_split_mode=ValSplitMode.FROM_TRAIN,
    seed=seed,
)

output_dir = models_dir / model_set
output_dir.mkdir(parents=True, exist_ok=True)
output_ckpt = output_dir / f"{angle}.ckpt"

engine = Engine(
    accelerator=accelerator,
    devices=1,
    default_root_dir=results_dir,
    logger=False,
    enable_progress_bar=False,
)

engine.fit(model=model, datamodule=datamodule)
engine.trainer.save_checkpoint(output_ckpt)

print("Saved checkpoint:", output_ckpt)


Accelerator: gpu
GPU: NVIDIA A100-SXM4-80GB


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

Selecting Coreset Indices.: 100%|██████████| 178789/178789 [14:36<00:00, 203.96it/s]
The validation set does not contain any anomalous images. As a result, the adaptive threshold will take the value of the highest anomaly score observed in the normal validation images, which may lead to poor predictions. For a more reliable adaptive threshold computation, please add some anomalous images to the validation set.
`Trainer.fit` stopped: `max_epochs=1` reached.
`weights_only` was not set, defaulting to `False`.


Saved checkpoint: working/models/RembgAlignedPatchcore/G01.ckpt
